# 1.0 - Preprocesamiento de datos

**MITSUI & CO. Commodity Prediction Challenge** (Kaggle).

Objetivo del notebook: explorar y preprocesar las series de precios de commodities
(`train.csv`, 557 features) y sus etiquetas (`train_labels.csv`, 424 targets), y dejar
listos `X.parquet` / `y.parquet` en `data/processed/`.

Las rutas se resuelven con `src/config.py` (lee `RAW_DATA_DIR` desde `.env`).

In [1]:
import sys
from pathlib import Path

# Hacer importable el paquete src/ desde notebooks/
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data.make_dataset import load_raw, preprocess, build_dataset

config.RAW_DATA_DIR, config.RAW_DATA_DIR.exists()

(WindowsPath('D:/DataScienceCompetitions/MITSUI&CO.Commodity_Prediction_Challenge/mitsui-commodity-prediction-challenge'),
 True)

## 1) Cargar datos crudos

In [2]:
features, labels, pairs = load_raw()
print('features:', features.shape, '| labels:', labels.shape, '| pairs:', pairs.shape)
features.head()

features: (1917, 558) | labels: (1917, 425) | pairs: (424, 3)


,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
0,0,2264.5,7205.0,2570.0,3349.0,NaN,NaN,NaN,NaN,NaN,...,1.699987,0.776874,0.888115,0.689954,0.066653,0.090582,0.119630,0.078135,13.822740,0.059163
1,1,2228.0,7147.0,2579.0,3327.0,NaN,NaN,NaN,NaN,NaN,...,1.695279,0.778682,0.889488,0.692628,0.067354,0.091297,0.120520,0.079066,13.888146,0.059895
2,2,2250.0,7188.5,2587.0,3362.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,1.692724,0.780186,0.894004,0.697490,0.067394,0.091478,0.120809,0.079287,13.983675,0.060037
3,3,2202.5,7121.0,2540.0,3354.0,4728.0,4737.0,4729.0,3430.0,3426.0,...,1.683111,0.785329,0.889439,0.698502,0.067639,0.091558,0.121021,0.079285,14.035571,0.059983
4,4,2175.0,7125.0,2604.0,3386.0,NaN,NaN,NaN,NaN,NaN,...,1.684816,0.787264,0.891042,0.701485,0.067443,0.091266,0.121055,0.078925,14.013760,0.059503


In [3]:
# Mapa de targets: cada target es un instrumento o un spread (A - B) a cierto lag
pairs.head()

,target,lag,pair
0,target_0,1,US_Stock_VT_adj_close
1,target_1,1,LME_PB_Close - US_Stock_VT_adj_close
2,target_2,1,LME_CA_Close - LME_ZS_Close
3,target_3,1,LME_AH_Close - LME_ZS_Close
4,target_4,1,LME_AH_Close - JPX_Gold_Standard_Futures_Close


## 2) Exploración rápida

In [4]:
# Valores faltantes por feature (instrumentos que no cotizan ciertos días)
na_ratio = features.drop(columns=[config.ID_COL]).isna().mean().sort_values(ascending=False)
na_ratio.head(10)

US_Stock_GOLD_adj_low                 0.870631
US_Stock_GOLD_adj_open                0.870631
US_Stock_GOLD_adj_high                0.870631
US_Stock_GOLD_adj_close               0.870631
US_Stock_GOLD_adj_volume              0.870631
JPX_Platinum_Mini_Futures_Low         0.059990
JPX_Gold_Mini_Futures_Open            0.059990
JPX_Gold_Rolling-Spot_Futures_Open    0.059990
JPX_Gold_Standard_Futures_Open        0.059990
JPX_Platinum_Mini_Futures_Open        0.059990
dtype: float64

In [5]:
# Distribución de lags de los targets
pairs['lag'].value_counts().sort_index()

lag
1    106
2    106
3    106
4    106
Name: count, dtype: int64

## 3) Preprocesamiento y guardado

`preprocess()` ordena por `date_id` e imputa faltantes con ffill/bfill.
`build_dataset()` persiste `X.parquet` / `y.parquet` en `data/processed/`.

In [6]:
X = preprocess(features)
print('NaN restantes en X:', int(X.drop(columns=[config.ID_COL]).isna().sum().sum()))
X.head()

NaN restantes en X: 0


,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
0,0,2264.5,7205.0,2570.0,3349.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,1.699987,0.776874,0.888115,0.689954,0.066653,0.090582,0.119630,0.078135,13.822740,0.059163
1,1,2228.0,7147.0,2579.0,3327.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,1.695279,0.778682,0.889488,0.692628,0.067354,0.091297,0.120520,0.079066,13.888146,0.059895
2,2,2250.0,7188.5,2587.0,3362.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,1.692724,0.780186,0.894004,0.697490,0.067394,0.091478,0.120809,0.079287,13.983675,0.060037
3,3,2202.5,7121.0,2540.0,3354.0,4728.0,4737.0,4729.0,3430.0,3426.0,...,1.683111,0.785329,0.889439,0.698502,0.067639,0.091558,0.121021,0.079285,14.035571,0.059983
4,4,2175.0,7125.0,2604.0,3386.0,4728.0,4737.0,4729.0,3430.0,3426.0,...,1.684816,0.787264,0.891042,0.701485,0.067443,0.091266,0.121055,0.078925,14.013760,0.059503


In [7]:
x_path, y_path = build_dataset()
print('Guardado:', x_path, y_path)

Guardado: D:\DataScienceResearchPeru\Machine_Learning_Engineering_2026_cohort_now\ProyectoFinal\Proyecto_ML_Engineering_DSRP_I_2026\data\processed\X.parquet D:\DataScienceResearchPeru\Machine_Learning_Engineering_2026_cohort_now\ProyectoFinal\Proyecto_ML_Engineering_DSRP_I_2026\data\processed\y.parquet
